# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashmita-21/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Baseline action rule

For this baseline, I use two signals to identify content that may need refreshing:

1. **Content age:** How many days have passed since the content was last updated.
2. **Recent impressions:** How many impressions the content received in the last 90 days.

Content that is at least 180 days old and has received at least 500 impressions in the last 90 days will be selected for a content refresh review.

This rule prioritizes older content that still receives visibility. Refreshing this content may help maintain its usefulness and performance.

**Reason codes:**
- `STALE_AND_VISIBLE`: Content meets both the age and impressions thresholds.
- `NO_TRIGGER`: Content does not meet both thresholds.

This is a simple baseline rule, not a prediction of future performance. The thresholds are initial assumptions and should be reviewed against the dataset.


In [4]:

import pandas as pd
import numpy as np
from pathlib import Path

# Locate the dataset
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/Flyrank-ML-Internship/data/raw/content_refresh_anonymized.csv")
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Check that content_refresh_anonymized.csv "
        "is available in the data/raw folder."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

# Confirm required columns exist
required_columns = ["content_age_days", "impressions_90d"]
missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns not found: {missing_columns}. "
        "Check the actual dataset column names."
    )

# Set baseline thresholds
AGE_THRESHOLD = 180
IMPRESSION_THRESHOLD = 500

# Apply the baseline rule
df["reason_code"] = np.where(
    (df["content_age_days"] >= AGE_THRESHOLD) &
    (df["impressions_90d"] >= IMPRESSION_THRESHOLD),
    "STALE_AND_VISIBLE",
    "NO_TRIGGER"
)

print("\nReason code counts:")
print(df["reason_code"].value_counts(dropna=False))

# Preview the selected records
display(df[[
    "content_age_days",
    "impressions_90d",
    "reason_code"
]].head(10))


Dataset loaded successfully.
Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Reason code counts:
reason_code
NO_TRIGGER           20071
STALE_AND_VISIBLE     9929
Name: count, dtype: int64


,content_age_days,impressions_90d,reason_code
0,187,3803,STALE_AND_VISIBLE
1,445,15320,STALE_AND_VISIBLE
2,141,12581,NO_TRIGGER
3,463,11751,STALE_AND_VISIBLE
4,263,19140,STALE_AND_VISIBLE
5,147,3970,NO_TRIGGER
6,90,20,NO_TRIGGER
7,445,1724,STALE_AND_VISIBLE
8,90,32574,NO_TRIGGER
9,257,1240,STALE_AND_VISIBLE


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


### Build the ranked action queue

After applying the baseline rule, I create an action queue to prioritize content that may benefit from a refresh.

The scoring process works as follows:

1. Records that meet both baseline conditions are assigned a refresh action.
2. Their impressions are used to calculate an action score. I use `log1p` to reduce the effect of very large impression values.
3. Records with higher scores are ranked first.
4. Records that do not meet the rule receive a score of zero and are not prioritized for immediate action.

The final ranked queue includes the content ID, action, score, reason code, and rank. I save the output as a CSV file for review.


In [5]:

# Create the action score
eligible = df["reason_code"] == "STALE_AND_VISIBLE"

df["action"] = np.where(
    eligible,
    "CONTENT_REFRESH_REVIEW",
    "NO_IMMEDIATE_ACTION"
)

# Calculate score using recent impressions
df["action_score"] = 0.0

if eligible.any():
    raw_scores = np.log1p(
        df.loc[eligible, "impressions_90d"].clip(lower=0)
    )

    max_score = raw_scores.max()

    if max_score > 0:
        df.loc[eligible, "action_score"] = (
            raw_scores / max_score
        ) * 100

# Sort records by score, then by impressions
df = df.sort_values(
    by=["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

# Assign rank
df["rank"] = np.arange(1, len(df) + 1)

# Select output columns
output_columns = [
    col for col in [
        "content_id",
        "content_age_days",
        "impressions_90d",
        "action",
        "action_score",
        "reason_code",
        "rank"
    ]
    if col in df.columns
]

ranked_queue = df[output_columns].copy()

# Save CSV
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"
ranked_queue.to_csv(OUTPUT_PATH, index=False)

print("Ranked queue created successfully.")
print("Output saved to:", OUTPUT_PATH)
print("\nAction counts:")
print(ranked_queue["action"].value_counts())

print("\nTop 10 ranked records:")
display(ranked_queue.head(10))


Ranked queue created successfully.
Output saved to: work/outputs/baseline_action_score.csv

Action counts:
action
NO_IMMEDIATE_ACTION       20071
CONTENT_REFRESH_REVIEW     9929
Name: count, dtype: int64

Top 10 ranked records:


,content_id,content_age_days,impressions_90d,action,action_score,reason_code,rank
0,content_5fe46e04994d,537,517715,CONTENT_REFRESH_REVIEW,100.000000,STALE_AND_VISIBLE,1
1,content_aaef01a50def,445,517109,CONTENT_REFRESH_REVIEW,99.991098,STALE_AND_VISIBLE,2
2,content_8c19996aa890,445,509252,CONTENT_REFRESH_REVIEW,99.874731,STALE_AND_VISIBLE,3
3,content_4c36c775b818,445,463103,CONTENT_REFRESH_REVIEW,99.152743,STALE_AND_VISIBLE,4
4,content_2dba2b1f9536,299,443434,CONTENT_REFRESH_REVIEW,98.822882,STALE_AND_VISIBLE,5
5,content_1a9e894be2e2,482,416180,CONTENT_REFRESH_REVIEW,98.340780,STALE_AND_VISIBLE,6
6,content_2c2606c5d176,362,347399,CONTENT_REFRESH_REVIEW,96.967816,STALE_AND_VISIBLE,7
7,content_db5989a78dd3,445,345111,CONTENT_REFRESH_REVIEW,96.917594,STALE_AND_VISIBLE,8
8,content_9532f197bbc8,445,309192,CONTENT_REFRESH_REVIEW,96.082283,STALE_AND_VISIBLE,9
9,content_8e7ba84a972b,224,288426,CONTENT_REFRESH_REVIEW,95.553875,STALE_AND_VISIBLE,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


### Review the top 20 recommendations

I review the first 20 records in the ranked action queue to understand which content items are being prioritized for refresh.

The review focuses on:
- The content's rank and action score.
- Its age and recent impressions.
- The reason code behind its selection.
- Whether the recommendation makes sense based on the available signals.

A high score indicates that the content meets the baseline conditions and has relatively strong recent visibility. However, the score is only a prioritization signal, not a guarantee that refreshing the content will improve performance.

Each recommendation should be checked before taking action.


In [6]:

# Select the top 20 records
top_20 = ranked_queue.head(20).copy()

# Add a confidence label based on the baseline rule
top_20["confidence"] = np.where(
    top_20["reason_code"] == "STALE_AND_VISIBLE",
    "Medium",
    "Low"
)

# Add a review note
top_20["review_note"] = np.where(
    top_20["reason_code"] == "STALE_AND_VISIBLE",
    "Check whether the content information is outdated and still relevant.",
    "Does not meet both baseline conditions; review only if other evidence exists."
)

# Display the top 20 review table
review_columns = [
    col for col in [
        "rank",
        "content_id",
        "content_age_days",
        "impressions_90d",
        "action_score",
        "action",
        "reason_code",
        "confidence",
        "review_note"
    ]
    if col in top_20.columns
]

display(top_20[review_columns])

print("Number of records reviewed:", len(top_20))
print("\nReason code distribution in top 20:")
print(top_20["reason_code"].value_counts())


,rank,content_id,content_age_days,impressions_90d,action_score,action,reason_code,confidence,review_note
0,1,content_5fe46e04994d,537,517715,100.000000,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
1,2,content_aaef01a50def,445,517109,99.991098,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
2,3,content_8c19996aa890,445,509252,99.874731,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
3,4,content_4c36c775b818,445,463103,99.152743,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
4,5,content_2dba2b1f9536,299,443434,98.822882,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
5,6,content_1a9e894be2e2,482,416180,98.340780,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
6,7,content_2c2606c5d176,362,347399,96.967816,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
7,8,content_db5989a78dd3,445,345111,96.917594,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
8,9,content_9532f197bbc8,445,309192,96.082283,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...
9,10,content_8e7ba84a972b,224,288426,95.553875,CONTENT_REFRESH_REVIEW,STALE_AND_VISIBLE,Medium,Check whether the content information is outda...


Number of records reviewed: 20

Reason code distribution in top 20:
reason_code
STALE_AND_VISIBLE    20
Name: count, dtype: int64


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak picks and leakage check

In this section, I examine weak recommendations and check whether the baseline scoring rule uses any information that would cause data leakage.

**Weak picks:**
- I inspect records with a zero action score.
- These records do not meet both baseline conditions.
- I check whether any potentially useful content was excluded by the rule.

**Leakage check:**
- I review the dataset columns for future-looking information.
- I avoid using future performance, outcome labels, or information that would not have been available when the recommendation was made.
- The baseline rule uses only content age and impressions from the previous 90 days.

This check helps ensure that the baseline is simple, understandable, and based on available historical information.


In [7]:

# 1. Inspect weak picks (records with zero action score)
weak_picks = ranked_queue[
    ranked_queue["action_score"] == 0
].copy()

print("Number of weak picks:", len(weak_picks))
print("\nSample weak picks:")
display(weak_picks.head(10))

# 2. Check whether any weak picks have high recent impressions
if "impressions_90d" in weak_picks.columns:
    print("\nWeak picks with at least 500 impressions:")
    display(
        weak_picks[
            weak_picks["impressions_90d"] >= 500
        ].head(10)
    )

# 3. Review column names for possible leakage
possible_leakage_terms = [
    "future",
    "next_",
    "outcome",
    "label",
    "target",
    "conversion_after",
    "trend_direction",
    "trend_pct",
    "is_declining"
]

possible_leakage_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in possible_leakage_terms)
]

print("\nColumns requiring leakage review:")
if possible_leakage_columns:
    print(possible_leakage_columns)
else:
    print("No obvious future-looking or target-like column names found.")

# 4. Confirm which columns the baseline actually uses
baseline_features = [
    "content_age_days",
    "impressions_90d"
]

print("\nFeatures used by the baseline:")
print(baseline_features)

# 5. Final leakage statement
print("\nLeakage check:")
print(
    "The baseline score uses content_age_days and impressions_90d only. "
    "Any flagged columns should be manually reviewed before using them "
    "in a future model."
)


Number of weak picks: 20071

Sample weak picks:


,content_id,content_age_days,impressions_90d,action,action_score,reason_code,rank
9929,content_2cb567c3c89b,153,497727,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9930
9930,content_44e481c8f55b,112,312694,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9931
9931,content_cb112fce36be,126,309910,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9932
9932,content_36ff89c8214e,144,295097,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9933
9933,content_b28d1efd668f,153,286608,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9934
9934,content_89e84d699e9e,154,275226,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9935
9935,content_aa4baf490b43,119,256290,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9936
9936,content_008fb02c46cb,111,236803,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9937
9937,content_813e88069237,153,233561,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9938
9938,content_ff94c9b6b411,154,228566,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9939



Weak picks with at least 500 impressions:


,content_id,content_age_days,impressions_90d,action,action_score,reason_code,rank
9929,content_2cb567c3c89b,153,497727,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9930
9930,content_44e481c8f55b,112,312694,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9931
9931,content_cb112fce36be,126,309910,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9932
9932,content_36ff89c8214e,144,295097,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9933
9933,content_b28d1efd668f,153,286608,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9934
9934,content_89e84d699e9e,154,275226,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9935
9935,content_aa4baf490b43,119,256290,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9936
9936,content_008fb02c46cb,111,236803,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9937
9937,content_813e88069237,153,233561,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9938
9938,content_ff94c9b6b411,154,228566,NO_IMMEDIATE_ACTION,0.0,NO_TRIGGER,9939



Columns requiring leakage review:
['trend_direction', 'trend_pct']

Features used by the baseline:
['content_age_days', 'impressions_90d']

Leakage check:
The baseline score uses content_age_days and impressions_90d only. Any flagged columns should be manually reviewed before using them in a future model.



## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.
